# ANFIS Hyperparameter Grid — Dashboard theo tỷ lệ train/test

Mỗi tỷ lệ **80/20, 70/30, 60/40** xuất **3 hình**:

| File | Nội dung |
|------|----------|
| `split_dashboard_*.png` | Confusion matrix train + test (config **tốt nhất**) |
| `split_loss_*.png` | Epoch vs Error — train lại với config **tốt nhất** (test acc cao nhất trong grid) |
| `hyperparam_accuracy_*.png` | Biểu đồ cột accuracy theo LR/epoch (từ grid search) |

**Bước 1:** `python run_hyperparam_grid.py` → CSV  
**Bước 2:** `python plot_split_dashboard.py` → 3 loại PNG / mỗi split

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from skanfis import scikit_anfis
from skanfis.fs import FS, LinguisticVariable, GaussianFuzzySet
from skanfis.experimental import RMSELoss

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 11,
})

In [ ]:
# --- Cau hinh ---
SPLIT_RATIOS = {"80-20": 0.2, "70-30": 0.3, "60-40": 0.4}
SPLIT_TITLES = {
    "80-20": "80 - 20% of train-test data",
    "70-30": "70 - 30% of train-test data",
    "60-40": "60 - 40% of train-test data",
}
LEARNING_RATES = [0.001, 0.01, 0.05]
EPOCH_LIST = [25, 50, 75, 100]
LR_COLORS = {0.001: "#f4a261", 0.01: "#8ecae6", 0.05: "#b39ddb"}
LR_LABELS = {0.001: "0.001", 0.01: "0.01", 0.05: "0.05"}

RUN_GRID_SEARCH = False
RESULTS_PATH = Path("models") / "hyperparam_grid_results.csv"
OUT_DIR = Path("models")

In [ ]:
# --- Tien xu ly WBCD (giong notebook training) ---
uci_url = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "breast-cancer-wisconsin/breast-cancer-wisconsin.data"
)
cols = [
    "sample_code_number", "clump_thickness", "uniformity_of_cell_size",
    "uniformity_of_cell_shape", "marginal_adhesion", "single_epithelial_cell_size",
    "bare_nuclei", "bland_chromatin", "normal_nucleoli", "mitoses", "class",
]
feature_cols = [
    "clump_thickness", "uniformity_of_cell_size", "uniformity_of_cell_shape",
    "marginal_adhesion", "single_epithelial_cell_size", "bare_nuclei",
    "bland_chromatin", "normal_nucleoli", "mitoses",
]
PAPER_TOP3_FEATURES = [
    "clump_thickness", "uniformity_of_cell_size", "uniformity_of_cell_shape",
]
FS_VAR_NAMES = {
    "clump_thickness": "ClumpThickness",
    "uniformity_of_cell_size": "CellSize",
    "uniformity_of_cell_shape": "CellShape",
}
LINGUISTIC_TERMS = ("low", "medium", "high")

df = pd.read_csv(uci_url, header=None, names=cols)
df = df.replace("?", np.nan).dropna().copy()
df["bare_nuclei"] = df["bare_nuclei"].astype(int)
df["target"] = (df["class"] == 4).astype(int)
X = df[feature_cols].values.astype(np.float32)
y = df["target"].values.astype(int)

OUTLIER_STD_MULTIPLIER = 2.5
data_center = X.mean(axis=0)
euclidean_distances = np.linalg.norm(X - data_center, axis=1)
outlier_threshold = euclidean_distances.mean() + OUTLIER_STD_MULTIPLIER * euclidean_distances.std()
keep_mask = euclidean_distances <= outlier_threshold
X_clean, y_clean = X[keep_mask], y[keep_mask]

scaler = StandardScaler()
X_norm = scaler.fit_transform(X_clean).astype(np.float32)
pca_full = PCA(n_components=9, random_state=42)
pca_full.fit(X_norm)

selected_idx = [feature_cols.index(c) for c in PAPER_TOP3_FEATURES]
X_selected = X_norm[:, selected_idx].astype(np.float32)

n_drop = len(X_selected) - 663
centroid_benign = X_selected[y_clean == 0].mean(axis=0)
centroid_malignant = X_selected[y_clean == 1].mean(axis=0)
dist_to_benign = np.linalg.norm(X_selected - centroid_benign, axis=1)
dist_to_malignant = np.linalg.norm(X_selected - centroid_malignant, axis=1)
own_dist = np.where(y_clean == 0, dist_to_benign, dist_to_malignant)
other_dist = np.where(y_clean == 0, dist_to_malignant, dist_to_benign)
quality_score = other_dist - own_dist
drop_idx = np.argsort(quality_score)[:n_drop]
quality_keep_mask = np.ones(len(X_selected), dtype=bool)
quality_keep_mask[drop_idx] = False
indices = np.where(quality_keep_mask)[0]
X_663 = X_selected[indices]
y_663 = y_clean[indices].astype(np.float32)

print(f"Dataset: {X_663.shape}, malignant rate={y_663.mean():.4f}")

In [ ]:
def build_fs_and_model(X_train, epochs):
    """ANFIS grid 27 luat, consequent hoc duoc (zerotype=False)."""
    fs = FS()
    for feat_col in PAPER_TOP3_FEATURES:
        fs_var = FS_VAR_NAMES[feat_col]
        col = X_train[:, PAPER_TOP3_FEATURES.index(feat_col)]
        col_min, col_max = float(col.min()), float(col.max())
        centers = np.linspace(col_min, col_max, 3).tolist()
        sigma = max((col_max - col_min) / 3.0, 1e-3)
        mf_low = GaussianFuzzySet(mu=centers[0], sigma=sigma, term="low")
        mf_med = GaussianFuzzySet(mu=centers[1], sigma=sigma, term="medium")
        mf_high = GaussianFuzzySet(mu=centers[2], sigma=sigma, term="high")
        fs.add_linguistic_variable(
            fs_var,
            LinguisticVariable([mf_low, mf_med, mf_high], concept=fs_var),
        )
    fs.set_crisp_output_value("out", 0)
    grid_rules = []
    for ct, cs, csh in itertools.product(LINGUISTIC_TERMS, repeat=3):
        grid_rules.append(
            f"IF (ClumpThickness IS {ct}) AND (CellSize IS {cs}) AND (CellShape IS {csh}) "
            f"THEN (out IS 0.5)"
        )
    fs.add_rules(grid_rules)
    return scikit_anfis(
        fs, description="WBCD_Gaussian27_GridSearch",
        epoch=epochs, hybrid=True, label="c", zerotype=False,
    )


def clamp_gaussian_sigma(model, min_sigma=1e-3):
    with torch.no_grad():
        for fuzzify_var in model.layer["fuzzify"].varmfs.values():
            for mf in fuzzify_var.mfdefs.values():
                if hasattr(mf, "sigma"):
                    mf.sigma.data.clamp_(min=min_sigma)


def train_model(model, X_train, y_train, epochs, learning_rate, return_history=False):
    optimizer = torch.optim.SGD(model.layer["fuzzify"].parameters(), lr=learning_rate)
    criterion = RMSELoss()
    X_train_t = torch.from_numpy(X_train).float()
    y_train_t = torch.from_numpy(y_train).float().unsqueeze(-1)
    history = []
    model.train()
    for ep in range(1, epochs + 1):
        with torch.no_grad():
            model.is_training = True
            model(X_train_t, y_train_t)
        model.is_training = False
        y_pred = model(X_train_t, y_train_t)
        optimizer.zero_grad()
        loss = criterion(y_pred, y_train_t)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.layer["fuzzify"].parameters(), 1.0)
        optimizer.step()
        clamp_gaussian_sigma(model)
        mse = torch.nn.functional.mse_loss(y_pred, y_train_t).item()
        rmse = float(np.sqrt(mse))
        history.append({"epoch": ep, "rmse": rmse, "error": rmse})
    model.eval()
    model.is_training = False
    if return_history:
        return model, pd.DataFrame(history)
    return model


def predict_binary(model, X):
    with torch.no_grad():
        y_pred = model(torch.from_numpy(X).float()).detach().numpy()
    return np.clip(np.round(y_pred), 0, 1).astype(int)


def best_config_for_split(grid_df, split_label):
    """lr + epoch co test accuracy cao nhat trong grid search."""
    sub = grid_df[grid_df["split_ratio"] == split_label]
    best = sub.loc[sub["test_accuracy"].idxmax()]
    return float(best["learning_rate"]), int(best["epochs"]), float(best["test_accuracy_pct"])

In [ ]:
# --- Chay grid search hoac doc ket qua da luu ---
if RUN_GRID_SEARCH or not RESULTS_PATH.exists():
    results = []
    total = len(SPLIT_RATIOS) * len(LEARNING_RATES) * len(EPOCH_LIST)
    run_idx = 0
    for split_label, test_size in SPLIT_RATIOS.items():
        X_tr, X_te, y_tr, y_te = train_test_split(
            X_663, y_663, test_size=test_size, random_state=42, stratify=y_663,
        )
        for lr in LEARNING_RATES:
            for n_epochs in EPOCH_LIST:
                run_idx += 1
                print(f"[{run_idx}/{total}] {split_label} lr={lr} ep={n_epochs}", flush=True)
                model = build_fs_and_model(X_tr, n_epochs)
                model = train_model(model, X_tr, y_tr, n_epochs, lr)
                acc = accuracy_score(y_te.astype(int), predict_binary(model, X_te))
                results.append({
                    "split_ratio": split_label,
                    "test_size": test_size,
                    "learning_rate": lr,
                    "epochs": n_epochs,
                    "test_accuracy": acc,
                    "test_accuracy_pct": round(acc * 100, 2),
                })
    grid_df = pd.DataFrame(results)
    RESULTS_PATH.parent.mkdir(exist_ok=True)
    grid_df.to_csv(RESULTS_PATH, index=False)
    print(f"Saved: {RESULTS_PATH}")
else:
    grid_df = pd.read_csv(RESULTS_PATH)
    print(f"Loaded: {RESULTS_PATH} ({len(grid_df)} rows)")

display(grid_df.sort_values(["split_ratio", "epochs", "learning_rate"]))

In [ ]:
CM_LABELS = [
    ("True Negative", (0, 0)),
    ("False Positive", (0, 1)),
    ("False Negative", (1, 0)),
    ("True Positive", (1, 1)),
]


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


def plot_confusion_matrix(ax, y_true, y_pred, title):
    """Heatmap confusion matrix giong paper (Benign/Malignant + TN/FP/FN/TP)."""
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    total = cm.sum()
    metrics = compute_metrics(y_true, y_pred)

    vmax = max(int(cm.max()), 1)
    sns.heatmap(
        cm,
        annot=False,
        fmt="d",
        cmap="YlOrRd",
        cbar=True,
        vmin=0,
        vmax=vmax,
        xticklabels=["Benign", "Malignant"],
        yticklabels=["Benign", "Malignant"],
        linewidths=2,
        linecolor="white",
        ax=ax,
    )
    for label, (r, c) in CM_LABELS:
        count = int(cm[r, c])
        pct = 100.0 * count / total if total else 0.0
        ax.text(
            c + 0.5, r + 0.5,
            f"{label}\n{count}\n{pct:.2f}%",
            ha="center", va="center", fontsize=10, color="black",
        )

    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)
    ax.set_xlabel("Predicted label", fontsize=11)
    ax.set_ylabel("True label", fontsize=11)
    ax.text(
        0.5, -0.22,
        (
            f"Accuracy = {metrics['accuracy']:.3f}    "
            f"Precision = {metrics['precision']:.3f}    "
            f"Recall = {metrics['recall']:.3f}    "
            f"F1 Score = {metrics['f1']:.3f}"
        ),
        transform=ax.transAxes,
        ha="center", va="top", fontsize=10,
    )
    return metrics


def plot_epoch_error(ax, history_df, title="Epoch vs Error (RMSE)"):
    ax.plot(history_df["epoch"], history_df["error"], color="#1f77b4", linewidth=2)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Error (RMSE)")
    ax.grid(True, alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def plot_grouped_accuracy(df, split_label, ax=None):
    sub = df[df["split_ratio"] == split_label].copy()
    bar_width = 0.22
    x = np.arange(len(EPOCH_LIST))
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 5.5))
    else:
        fig = ax.figure
    for i, lr in enumerate(LEARNING_RATES):
        vals = [
            float(sub[(sub["epochs"] == ep) & (sub["learning_rate"] == lr)]["test_accuracy_pct"].iloc[0])
            for ep in EPOCH_LIST
        ]
        offset = (i - (len(LEARNING_RATES) - 1) / 2) * bar_width
        bars = ax.bar(
            x + offset, vals, width=bar_width,
            color=LR_COLORS[lr], label=LR_LABELS[lr], edgecolor="none", alpha=0.92,
        )
        for bar, val in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                f"{val:.2f}", ha="center", va="bottom", rotation=90, fontsize=9,
            )
    ax.set_title(SPLIT_TITLES[split_label], fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Epoch at different learning rate", fontsize=11)
    ax.set_ylabel("Accuracy (%)", fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels([str(e) for e in EPOCH_LIST])
    ax.set_ylim(0, 105)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return fig, ax


def plot_split_dashboard(split_label, test_size, grid_df):
    """Moi ty le split: CM + loss (config tot nhat) + bieu do cot grid search."""
    lr, epochs, best_acc_pct = best_config_for_split(grid_df, split_label)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_663, y_663, test_size=test_size, random_state=42, stratify=y_663,
    )
    model = build_fs_and_model(X_tr, epochs)
    model, history_df = train_model(model, X_tr, y_tr, epochs, lr, return_history=True)
    y_tr_pred = predict_binary(model, X_tr)
    y_te_pred = predict_binary(model, X_te)
    slug = split_label.replace("-", "_")
    info = {
        "learning_rate": lr, "epochs": epochs, "best_acc_pct": best_acc_pct,
        "history": history_df, "slug": slug,
    }

    fig_cm, axes = plt.subplots(1, 2, figsize=(14, 6))
    plot_confusion_matrix(axes[0], y_tr.astype(int), y_tr_pred, "Confusion matrix for WDBC - train data")
    plot_confusion_matrix(axes[1], y_te.astype(int), y_te_pred, "Confusion matrix for WDBC - test data")
    fig_cm.suptitle(
        f"{SPLIT_TITLES[split_label]}  |  best: lr={lr}, epochs={epochs}, test acc={best_acc_pct:.2f}%",
        fontsize=14, fontweight="bold", y=1.02,
    )
    plt.tight_layout()

    best_idx = history_df["error"].idxmin()
    best_ep = int(history_df.loc[best_idx, "epoch"])
    best_rmse = float(history_df.loc[best_idx, "error"])
    fig_loss, ax_err = plt.subplots(figsize=(9, 5.5))
    ax_err.plot(history_df["epoch"], history_df["error"], color="#1f77b4", linewidth=2, label="Train RMSE")
    ax_err.scatter([best_ep], [best_rmse], color="#2ca02c", s=80, zorder=5,
                   label=f"Best epoch={best_ep}, RMSE={best_rmse:.4f}")
    ax_err.set_title(
        f"Epoch vs Error (RMSE) — best config\n"
        f"{SPLIT_TITLES[split_label]} | lr={lr}, epochs={epochs}, test acc={best_acc_pct:.2f}%",
        fontweight="bold",
    )
    ax_err.set_xlabel("Epoch")
    ax_err.set_ylabel("Error (RMSE)")
    ax_err.grid(True, alpha=0.3)
    ax_err.legend(loc="best")
    ax_err.spines["top"].set_visible(False)
    ax_err.spines["right"].set_visible(False)
    plt.tight_layout()

    fig_bar, _ = plot_grouped_accuracy(grid_df, split_label)
    plt.tight_layout()

    return fig_cm, fig_loss, fig_bar, info

In [ ]:
# --- Moi ty le split: 3 hinh (CM + loss best config + bieu do cot) ---
for split_label, test_size in SPLIT_RATIOS.items():
    fig_cm, fig_loss, fig_bar, info = plot_split_dashboard(split_label, test_size, grid_df)
    for fig in (fig_cm, fig_loss, fig_bar):
        plt.figure(fig.number)
        plt.show()

    slug = info["slug"]
    paths = {
        "CM": OUT_DIR / f"split_dashboard_{slug}.png",
        "loss": OUT_DIR / f"split_loss_{slug}.png",
        "bar": OUT_DIR / f"hyperparam_accuracy_{slug}.png",
    }
    fig_cm.savefig(paths["CM"], dpi=200, bbox_inches="tight")
    fig_loss.savefig(paths["loss"], dpi=200, bbox_inches="tight")
    fig_bar.savefig(paths["bar"], dpi=200, bbox_inches="tight")
    plt.close(fig_cm)
    plt.close(fig_loss)
    plt.close(fig_bar)
    print(
        f"Saved: {paths['CM']}\n"
        f"Saved: {paths['loss']}  (best lr={info['learning_rate']}, epochs={info['epochs']})\n"
        f"Saved: {paths['bar']}"
    )

In [ ]:
# Bieu do cot accuracy da duoc luu trong cell tren (hyperparam_accuracy_*.png)